<a href="https://colab.research.google.com/github/Biteo326/blank-app/blob/main/NLP_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
# Install the required Libraries
!pip install pandas deep-translator vaderSentiment

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 1.4 MB/s eta 0:00:00


In [18]:
# Import libraries
import pandas as pd
import numpy as np
import re
from deep_translator import GoogleTranslator
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

In [19]:
# Load Dataset
df = pd.read_csv('/content/sample_data/Dataset.csv') #Change the directory if need

print("Dataset Shape:", df.shape)
df.head()

Dataset Shape: (606, 19)


,User Id,User Shop Id,User Name,Anonymous,Comment Id,Comment Date,Comment,Rating,Detail Rating,Comment Images,Comment Videos,Bought Products,Product Id,Product Url,Product Name,Product Image,Shop Id,Region,Scraped At
0,16240690,NaN,liendamohd,No,32342828916,2025-03-07T02:14:36.000Z,Performance:Excellent\nQuality:Excellent\n\nFa...,5,product_quality: 5\nseller_service: 5\ndeliver...,https://down-bs-us.img.susercontent.com/my-111...,https://down-zl-my.vod.susercontent.com/api/v4...,TWS Wireless Transparent In-Ear Earphones | LE...,24431177751,https://shopee.com.my/TWS-Wireless-Transparent...,TWS Wireless Transparent In-Ear Earphones | LE...,https://down-bs-us.img.susercontent.com/my-111...,12609016,MY,2026-06-13T18:46:18.094Z
1,1411611864,NaN,_norazl1nn,No,30268997432,2025-02-06T06:48:56.000Z,Quality:very good!\nPerformance:also good\n\nF...,5,product_quality: 5\nseller_service: 5\ndeliver...,https://down-bs-us.img.susercontent.com/my-111...,https://down-aka-my.vod.susercontent.com/api/v...,TWS Wireless Transparent In-Ear Earphones | LE...,24431177751,https://shopee.com.my/TWS-Wireless-Transparent...,TWS Wireless Transparent In-Ear Earphones | LE...,https://down-bs-us.img.susercontent.com/my-111...,12609016,MY,2026-06-13T18:46:18.094Z
2,372240270,NaN,snowy64,No,32073161727,2025-03-04T12:34:16.000Z,Quality:Good\nPerformance:Sound very clear and...,5,product_quality: 5\nseller_service: 5\ndeliver...,https://down-bs-us.img.susercontent.com/my-111...,https://down-zl-my.vod.susercontent.com/api/v4...,TWS Wireless Transparent In-Ear Earphones | LE...,24431177751,https://shopee.com.my/TWS-Wireless-Transparent...,TWS Wireless Transparent In-Ear Earphones | LE...,https://down-bs-us.img.susercontent.com/my-111...,12609016,MY,2026-06-13T18:46:18.094Z
3,171946168,NaN,azham33,No,36158289859,2025-04-08T10:44:56.000Z,Performance:Padu\nQuality:Kecik dan gempak\n\n...,5,product_quality: 5\nseller_service: 5\ndeliver...,https://down-bs-us.img.susercontent.com/my-111...,NaN,TWS Wireless Transparent In-Ear Earphones | LE...,24431177751,https://shopee.com.my/TWS-Wireless-Transparent...,TWS Wireless Transparent In-Ear Earphones | LE...,https://down-bs-us.img.susercontent.com/my-111...,12609016,MY,2026-06-13T18:46:18.094Z
4,1106989690,NaN,b*****8,Yes,19965333628,2024-11-30T03:36:35.000Z,Performance:good\nQuality:good\n\nThe packagin...,5,product_quality: 5\nseller_service: 5\ndeliver...,https://down-bs-us.img.susercontent.com/my-111...,https://down-zl-my.vod.susercontent.com/api/v4...,TWS Wireless Transparent In-Ear Earphones | LE...,24431177751,https://shopee.com.my/TWS-Wireless-Transparent...,TWS Wireless Transparent In-Ear Earphones | LE...,https://down-bs-us.img.susercontent.com/my-111...,12609016,MY,2026-06-13T18:46:18.094Z


In [20]:
# Check Column Names
print(df.columns)

Index(['User Id', 'User Shop Id', 'User Name', 'Anonymous', 'Comment Id',
       'Comment Date', 'Comment', 'Rating', 'Detail Rating', 'Comment Images',
       'Comment Videos', 'Bought Products', 'Product Id', 'Product Url',
       'Product Name', 'Product Image', 'Shop Id', 'Region', 'Scraped At'],
      dtype='object')


In [21]:
# Keep only the columns of Comments
df = df[['Comment']]

print("Rows before cleaning:", len(df))
df.head()

Rows before cleaning: 606


,Comment
0,Performance:Excellent\nQuality:Excellent\n\nFa...
1,Quality:very good!\nPerformance:also good\n\nF...
2,Quality:Good\nPerformance:Sound very clear and...
3,Performance:Padu\nQuality:Kecik dan gempak\n\n...
4,Performance:good\nQuality:good\n\nThe packagin...


In [22]:
# Remove Empty Comments

df = df.dropna(subset=['Comment'])

df = df[df['Comment'].astype(str).str.strip() != '']

print("Rows after removing empty comments:", len(df))

Rows after removing empty comments: 591


In [23]:
# Text Cleaning Function

def clean_text(text):

    text = str(text)

    # lowercase
    text = text.lower()

    # remove urls
    text = re.sub(r'http\S+|www\S+', '', text)

    # remove common Shopee review attributes
    text = re.sub(r'performance:.*?\n', '', text)
    text = re.sub(r'quality:.*?\n', '', text)

    # remove emojis
    text = re.sub(
        "["
        "\U0001F600-\U0001F64F"
        "\U0001F300-\U0001F5FF"
        "\U0001F680-\U0001F6FF"
        "\U0001F1E0-\U0001F1FF"
        "]+",
        "",
        text,
        flags=re.UNICODE
    )

    # remove numbers
    text = re.sub(r'\d+', '', text)

    # remove punctuation
    text = re.sub(r'[^\w\s]', ' ', text)

    # remove extra spaces
    text = re.sub(r'\s+', ' ', text)

    return text.strip()

In [24]:
# Apply Cleaning

df['Cleaned_Comment'] = df['Comment'].apply(clean_text)

df = df[df['Cleaned_Comment'] != '']

print("Rows after text cleaning:", len(df))

Rows after text cleaning: 591


In [25]:
# Translate the languages to English

def translate_to_english(text):

    try:
        translated = GoogleTranslator(
            source='auto',
            target='en'
        ).translate(text)

        return translated

    except:
        return text

In [26]:
# Apply the translator

from tqdm import tqdm

tqdm.pandas()

df['English_Comment'] = df['Cleaned_Comment'].progress_apply(
    translate_to_english
)

100%|██████████| 591/591 [03:55<00:00,  2.51it/s]


In [27]:
# Sentiment Analysis

analyzer = SentimentIntensityAnalyzer()

def get_sentiment(text):

    score = analyzer.polarity_scores(text)['compound']

    if score >= 0.05:
        return 'Positive'

    elif score <= -0.05:
        return 'Negative'

    else:
        return 'Neutral'

In [28]:
# Generate Sentiment Labels

df['Sentiment'] = df['English_Comment'].apply(
    get_sentiment
)

In [29]:
# View Sample Results

df[['Comment',
    'Cleaned_Comment',
    'English_Comment',
    'Sentiment']].head(20)

,Comment,Cleaned_Comment,English_Comment,Sentiment
0,Performance:Excellent\nQuality:Excellent\n\nFa...,fast delivery long lasting battery worth the p...,fast delivery long lasting battery worth the p...,Positive
1,Quality:very good!\nPerformance:also good\n\nF...,fast delivery i ordered this a few times even ...,fast delivery i ordered this a few times even ...,Positive
2,Quality:Good\nPerformance:Sound very clear and...,packaging bagus dan penghantaran cepat sangat ...,"good packaging and fast delivery, very beautif...",Positive
3,Performance:Padu\nQuality:Kecik dan gempak\n\n...,senang dibawak dan tak mudah tercabut padu pad...,easy to carry and not easy to tear off because...,Positive
4,Performance:good\nQuality:good\n\nThe packagin...,the packaging is handled in good condition the...,the packaging is handled in good condition the...,Positive
5,Quality:sangat padu\nPerformance:bass mantap g...,macam tak bercaya harag murah bunyi padu desig...,"I can't believe the price is cheap, the sound ...",Positive
6,Quality:Good\n\noklaa kena dengan harga sbb na...,oklaa kena dengan harga sbb nak buat dgr lagu ...,It's ok with the price because I want to do it...,Positive
7,Performance:good\nQuality:good\n\nDh test sala...,dh test salah yg sy beli mmg best dia punya so...,"I tested the wrong one that I bought, it's the...",Positive
8,"Performance:white,black\nQuality:best giler…ba...",fast delivery memang sedap ii gile pakai earbu...,"fast delivery, it's really good, I love using ...",Positive
9,Quality:Good\nPerformance:Nicee\n\nAirpods tha...,airpods that have a silicone part that can pro...,airpods that have a silicone part that can pro...,Positive


In [30]:
# Check Sentiment Distribution

print(df['Sentiment'].value_counts())

Sentiment
Positive    310
Negative    183
Neutral      98
Name: count, dtype: int64


In [31]:
# Save Cleaned Dataset

df.to_csv(
    'Final_Cleaned_Dataset.csv',
    index=False
)

print("Dataset saved successfully!")

Dataset saved successfully!


In [32]:
# Download Cleaned Dataset

from google.colab import files

files.download('Final_Cleaned_Dataset.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>